# Work with Embeddings: Analyse

This notebook explores possibilities to profit from embeddings of a collection of sources. The collection is the digital collection of sources of the project [The School of Salamanca](https://salamanca.school/), and the notebook uses multilingual embeddings from OpenAI, Cohere and, eventually, Ollama.

## 0. Preliminaries

In [ ]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

### 1.2 Load libraries

In [ ]:
import os
import logging
import dotenv
import polars as pl
import json
from tqdm.notebook import tqdm
from time import sleep
import numpy as np
import pickle
from typing import Any, Dict, List, Optional
import urllib.parse
import requests
import glob
from pathlib import Path
from collections import deque
from datetime import datetime, timedelta

### 1.3 Configuration

#### 1.3.1 File paths

In [ ]:
# Input files
file_path_in = './out-data'

# These files will be auto-detected from the most recent timestamp
# Priority order: parquet > pickle > csv (for docs)
# For embeddings: parquet > pickle

# We'll search for files with this pattern:
# YYYY-MM-DD_*_docs.{parquet,pkl,csv}
# YYYY-MM-DD_*_embeddings.{parquet,pkl}
# YYYY-MM-DD_*_processing_metadata.json
# YYYY-MM-DD_*_embedding_statistics.json

file_path_seed_words = './in-data/lemmata_20240322.csv'
file_path_stop_words = './in-data/stopwords-la-new-line_27.01.22.txt'


#### 1.3.2 Limits

In [ ]:
# Here we set the number of documents (paragraphs) to process
# Set it to a lower value until everything runs well, then increase it
# Set it to -1 to process all documents:
max_documents=-1

#### 1.3.3 Topic Modeling Configuration

In [ ]:
min_topic_size = 20   # how many documents must feature a topic for the topic to be legitimate
top_n_words = 35      # how many words to display in topic representation

## 2. Utility Functions

### 2.1 Logging Configuration

Configure structured logging for the embedding generation process.

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

## 3. Read Input Data

### 3.1 Read seed/stop words

Next, we read our seed words and stop words. For seed words, we take the lemmas of the project dictionary. In fact, the seed words list is an array, each item of which contains an array with potentially multiple, related words.

In [ ]:
with open(file_path_seed_words) as seed_words_file:
    seed_words = [line.rstrip().split(",") for line in seed_words_file]

stop_words = []
with open(file_path_stop_words) as stop_words_file:
    for line in stop_words_file:
        stop_words.append(line.strip())

Give some information about seed/stop words

In [ ]:
print(f"Number of seed words: {len(seed_words)}")
print(seed_words[:10])
print()
print(f"Number of stop words: {len(stop_words)}")
print(stop_words[:10])

### 3.2 Load document and embedding data

In [ ]:
def find_latest_files(directory: str, pattern: str) -> Optional[str]:
    """Find the most recent file matching the pattern."""
    files = glob.glob(os.path.join(directory, pattern))
    if not files:
        return None
    # Sort by modification time, most recent first
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

def load_docs_data(directory: str) -> pl.DataFrame:
    """Load docs data from parquet, pickle, or CSV, in that order."""
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_docs.parquet")
    if parquet_file:
        print(f"Loading docs from parquet: {parquet_file}")
        return pl.read_parquet(parquet_file)
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_docs.pkl")
    if pickle_file:
        print(f"Loading docs from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            df = pickle.load(f)
            # If it's a pandas DataFrame, convert to polars
            if hasattr(df, 'to_dict'):  # pandas DataFrame check
                return pl.from_pandas(df)
            return df
    
    # Try CSV
    csv_file = find_latest_files(directory, "*_docs.csv")
    if csv_file:
        print(f"Loading docs from CSV: {csv_file}")
        return pl.read_csv(csv_file)
    
    raise FileNotFoundError(f"No docs files found in {directory}")

def load_embeddings_data(directory: str) -> dict:
    """Load embeddings data from parquet or pickle, in that order.
    Returns a nested dict: {provider_name: {doc_id: [embedding_vector]}}
    """
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_embeddings.parquet")
    if parquet_file:
        print(f"Loading embeddings from parquet: {parquet_file}")
        # The parquet was created from a nested dict: {provider: {doc_id: embedding}}
        # When saved with pl.DataFrame(nested_dict).write_parquet(), 
        # it creates columns named after providers, with each cell containing the doc_id->embedding mapping
        df_embeddings = pl.read_parquet(parquet_file)
        
        # Convert back to nested dict structure
        # Each column represents a provider, column values are the {doc_id: embedding} dicts
        embeddings_dict = {}
        for column_name in df_embeddings.columns:
            # Get the provider's embeddings (should be a single row with a dict/struct)
            provider_data = df_embeddings[column_name][0]
            if provider_data is not None:
                embeddings_dict[column_name] = provider_data
        
        return embeddings_dict
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_embeddings.pkl")
    if pickle_file:
        print(f"Loading embeddings from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            return pickle.load(f)
    
    raise FileNotFoundError(f"No embeddings files found in {directory}")

def load_metadata(directory: str) -> dict:
    """Load processing metadata JSON."""
    metadata_file = find_latest_files(directory, "*_processing_metadata.json")
    if metadata_file:
        print(f"Loading metadata from: {metadata_file}")
        with open(metadata_file, "r") as f:
            return json.load(f)
    return {}

# Load all data
print("="*80)
print("Loading data files...")
print("="*80)

docs = load_docs_data(file_path_in)
embeddings_dict = load_embeddings_data(file_path_in)
metadata = load_metadata(directory=file_path_in)

print(f"\nLoaded {docs.height} documents")
print(f"Loaded embeddings for {len(embeddings_dict)} providers:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  - {provider}: {len(provider_embeddings)} documents")

if metadata:
    print(f"\nProcessing metadata:")
    print(f"  Processing date: {metadata.get('processing_date', 'N/A')}")
    if 'providers' in metadata:
        print(f"  Providers in metadata: {list(metadata['providers'].keys())}")

#### 3.2.1 Data Structure Overview

The loaded data has the following structure:
- **docs**: Polars DataFrame with document metadata (url, content, xmlid, author, etc.)
- **embeddings_dict**: Nested dictionary organized as:
  ```python
  {
      "provider_name_1": {
          "doc_id_1": [embedding_vector_1],
          "doc_id_2": [embedding_vector_2],
          ...
      },
      "provider_name_2": {
          "doc_id_1": [embedding_vector_1],
          ...
      }
  }
  ```
- **metadata**: Configuration and processing information from the embedding creation run

Now, give some information about the data:

In [ ]:
print("="*80)
print("Data Overview")
print("="*80)
print(f"\nShape of docs dataframe: {docs.shape}")
print(f"Number of available documents: {docs.height}")
print(f"\nDocument length statistics:")
if "len_content" in docs.columns:
    print(docs["len_content"].describe())
else:
    print("  (len_content column not available)")

print("\nEmbeddings by provider:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  {provider}: {len(provider_embeddings)} embeddings")
    # Show embedding dimension
    if provider_embeddings:
        sample_embedding = next(iter(provider_embeddings.values()))
        print(f"    - Dimension: {len(sample_embedding)}")

print("\nFirst 3 rows of docs dataframe:")
docs.head(3)

## 4. Fit (Train) Topic Model

We split the remaining BERTopic steps between those that are part of the preparation of the (training of the) topic model, and those that affect the topic representationse: Dimensionality Reduction and Clustering on the one hand, Vectorization, c-TF-IDF representation on the other.

For all of these, we supply the default algorithms, but establish code to be able to easily change any of them in the future.

In [ ]:
# Define BERTopic parameters
# For helpful comments about the parameters,
# see https://medium.com/grabngoinfo/hyperparameter-tuning-for-bertopic-model-in-python-104445778347

# A. Dimensionality Reduction: umap_model

# Initialize the UMAP model for dimensionality reduction
dimred_model = umap.UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine', random_state=42)
# dimred_model = umap.UMAP()

# If we should want to not do any dimensionality reduction, we can use this:
# from bertopic.dimensionality import BaseDimensionalityReduction
# dimred_model = BaseDimensionalityReduction()

# B. Clustering

cluster_model = HDBSCAN(min_cluster_size=20, metric='euclidean', cluster_selection_method='eom', prediction_data=True)


Do the actual BERTopic model initialization and fitting:

In [ ]:
topic_model = BERTopic(
    umap_model=dimred_model,       # yes, the parameter name is called like the default method
    hdbscan_model=cluster_model,   # yes, the parameter name is called like the default method
    min_topic_size=min_topic_size,
    top_n_words=top_n_words,
    calculate_probabilities=True,
    verbose=True
)

tm_loaded = False
if os.path.isdir(file_path_tm_cache):
    try:
        topic_model = BERTopic.load(file_path_tm_cache)
    except Exception as e:
        print("Could not load topic model although a cache dir is present. Error: " + e)
        print("Fitting new topic model instead...")
        topics, probs = topic_model.fit_transform(paragraphs, embeddings)
    else:
        topics, probs = topic_model.transform(paragraphs, embeddings)
        tm_loaded = True
else:
    topics, probs = topic_model.fit_transform(paragraphs, embeddings)

print("\n")

### Interpret results

Get some diagnostic sample from the topics and probabilites - first, without finetuning the topic representation.

In [ ]:
topics_count = len(topic_model.topic_labels_)
topics_count

In [ ]:
df_topic_freq = topic_model.get_topic_freq()
df_topic_freq

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)

In [ ]:
topic_model.get_document_info(paragraphs)

## Tune Topic Representations

Now, if the above looks halfway reasonable, continue with work on improving the topic representation. This should work without retraining the model ...

### Vectorizer & c-TF/IDF Parameters

First, define vectorizer and tf/idf models, incorporating stopwords and seed words lists:

In [ ]:
# Topic representation (a): Vectorization

# Again, for helpful comments on parameters,
# see https://medium.com/grabngoinfo/hyperparameter-tuning-for-bertopic-model-in-python-104445778347

# min_df (minimum document frequency): in how many documents must a word occur for it to be eligible for topic representation
# max_features: how large is the vocabulary of words to consider

from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words=stop_words, ngram_range=(1, 3), min_df=6, max_features=100_000)

In [ ]:
# Topic representation (b): c-TF-IDF representation

# seed_multiplier: what to multiply the TF/IDF score of a word with when it is mentioned in the seed word list

from bertopic.vectorizers import ClassTfidfTransformer
ctfidf_model = ClassTfidfTransformer(seed_words=seed_words, seed_multiplier=10, reduce_frequent_words=True)

### Topic Labels and Descriptions

In [ ]:
# find the .env file and load it
load_dotenv(find_dotenv())

# get our API keys
oai_api_key = getenv("OPENAI_API_KEY")
coh_api_key = getenv("COHERE_API_KEY")

# KeyBERT
from bertopic.representation import KeyBERTInspired
keybert_model = KeyBERTInspired()

# OpenAI Labels
from bertopic.representation import OpenAI
import openai
oai_client = openai.OpenAI(api_key=oai_api_key)
oai_label_model = OpenAI(oai_client, model="gpt-4o", delay_in_seconds=10, chat=True)

# OpenAI Summaries
summarization_prompt = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a summarizing description of this topic:

"""
oai_sum_model = OpenAI(oai_client, model="gpt-4o", chat=True, prompt=summarization_prompt, nr_docs=5, delay_in_seconds=10)

# Cohere
import cohere
from bertopic.representation import Cohere
coh_client = cohere.Client(coh_api_key)
coh_label_model = Cohere(coh_client, model="command-r-plus", delay_in_seconds=15)

# Cohere Summaries
coh_sum_model = Cohere(coh_client, model="command-r-plus", prompt=summarization_prompt, nr_docs=5, delay_in_seconds=20)

representation_model = {
    "Main":  oai_label_model,
    "Cohere-Label": coh_label_model,
    # "KeyBERT": keybert_model,
    "GPT-Summary":  oai_sum_model,
    # "Cohere-Summary": coh_sum_model
}

In [ ]:
if not tm_loaded:
    topic_model.update_topics(paragraphs, vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model, top_n_words=top_n_words, representation_model=representation_model)

### Lemmatizing?

Something to try out. It's not clear how much improvement it brings. It's clear however, that it has to come after creating the embeddings. And if we want to create labels or summaries with a LLM, we need to do it before lemmatizing everything, too...

In [ ]:
# TODO:
# You can hook into Vectorization to do lemmatiziation:
# cf. https://github.com/MaartenGr/BERTopic/issues/286
#
# class LemmaTokenizer:
#    def __init__(self):
#        self.wnl = WordNetLemmatizer()
#    def __call__(self, doc):
#        return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]
#
# vectorizer_model= CountVectorizer(tokenizer=LemmaTokenizer())
#
# topic_model = BERTopic(vectorizer_model=vectorizer_model)

## Interpret (new) results

Now, get new diagnostics.

In [ ]:
df_topic_freq = topic_model.get_topic_freq()
df_topic_freq

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(1, full=True)

In [ ]:
if not os.path.exists('./figs'):
    os.makedir('./figs')

In [ ]:
fig_barchart = topic_model.visualize_barchart(top_n_topics=topics_count, n_words=8, custom_labels=True)
fig_barchart.write_html("figs/fig_barchart.html")
# fig_barchart

In [ ]:
fig_topics = topic_model.visualize_topics()
fig_topics.write_html("figs/fig_topics.html")
# fig_topics

In [ ]:
# Topic Similarity Matrix
fig_heatmap = topic_model.visualize_heatmap(custom_labels=True)
fig_heatmap.write_html("figs/fig_heatmap.html")
# fig_heatmap

In [ ]:
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(paragraphs, reduced_embeddings=reduced_embeddings)

In [ ]:
# Check if have a good cut-off of top_n_words
fig_termrank = topic_model.visualize_term_rank()
fig_termrank.write_html("figs/fig_termrank.html")
# fig_termrank

## Save results

In [ ]:
# Export our topics to a new csv file, too
topic_model.get_topic_info().to_csv(file_path_tm_export, index=False)

# And save the topic model with all sub-models to a pickle file
print("Saving Topic Model to disk ...")
topic_model.save(file_path_tm_cache, serialization="safetensors", save_ctfidf=True, save_embedding_model=False)

# with open(file_path_tm_cache + ".pkl", "wb") as fOut:
#    pickle.dump(topic_model, fOut)
# print("Done.")

In [ ]:
# Save the generated topics for all the documents to the dataframe
docs['topic'] = topic_model.topics_

In [ ]:
# Display the DataFrame
print("\nFirst rows of docs dataframe:")
docs[:3]

In [ ]:
# Export the documents data frame to a new csv file
docs.to_csv(file_path_export, index=False)

## Next steps

See the discussion in this cookbook: [Nick T.: "Topic Modeling with BERTopic: A Cookbook with an End-to-end Example (Part 2)"](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-2-1ae591f76a25).

<div class="alert alert-warning"><b>After working and getting good results in the following steps, do not forget to update the data frame and save it to CSV again.</b></div>

### Topics per Class (author, year etc.)

#### authors

In [ ]:
topics_per_author = topic_model.topics_per_class(paragraphs, classes=docs['author'], global_tuning=False)
fig_tpa = topic_model.visualize_topics_per_class(topics_per_author, top_n_topics=6, normalize_frequency=True, custom_labels=True)
fig_tpa.write_html("figs/fig_tpa.html")
# fig_tpa

#### years

In [ ]:
topics_per_year = topic_model.topics_per_class(paragraphs, classes=docs['year'], global_tuning=False)
fig_tpy = topic_model.visualize_topics_per_class(topics_per_year, top_n_topics=6, normalize_frequency=True, custom_labels=True)
fig_tpy.write_html("figs/fig_tpy.html")
# fig_tpy

### Outlier Reduction?

see [Outliers Reduction](https://medium.com/@nick-tan/topic-modeling-with-bertopic-a-cookbook-with-an-end-to-end-example-part-1-3ef739b8d9f8#8aaf) in the cookbook.

In [ ]:
# rinse and repeat

### ("Manual") Topic Merging

In [ ]:
# are there topics we would prefer to have merged? It goes like this:
# topics_to_merge = [
#    [10, 11]
# ]
# topic_model.merge_topics(paragraphs, topics_to_merge)

### Understand why a particular document is assigned to its topic

...

In [ ]:
topic_distr, topic_token_distr = topic_model.approximate_distribution(paragraphs, calculate_tokens=True, use_embedding_model=True)

In [ ]:
example_doc_id = 426
fig_ad = topic_model.visualize_approximate_distribution(paragraphs[example_doc_id], topic_token_distr[example_doc_id])
fig_ad.write_html("figs/fig_ad.html")
# fig_ad

In [ ]:
# To visualize the topic distributions in a document
# topic_model.visualize_distribution(topic_distr[example_doc_id], min_probability=0.00002)
# Visualize probability distribution
fig_dist = topic_model.visualize_distribution(topic_model.probabilities_[example_doc_id], min_probability=0.015, custom_labels=True)
fig_dist.write_html("figs/fig_dist.html")
# fig_dist

In [ ]:
# Topic Hierarchy
fig_hier = topic_model.visualize_hierarchy()
fig_hier.write_html("figs/fig_hier.html")
# fig_hier

Either assign labels "manually" by interpreting them intellectually. Or ask ChatGPT (or another LLM) to produce a label and description based on the most salient words and documents...

In [ ]:
# Add topic labels and descriptions
topic_labels_dict = {}
topic_descriptions_dict = {}
docs['topic_label'] = docs.topic.map(topic_labels_dict)
docs['topic_description'] = docs.topic.map(topic_descriptions_dict)

## Dynamic Topic Modeling

Next thing to try: [*dynamic* topic modeling](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)

In [ ]:
years = docs['year']
topics_over_time = topic_model.topics_over_time(paragraphs, years, evolution_tuning=True, global_tuning=True)

In [ ]:
fig_tot = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=10)
fig_tot.write_html("figs/fig_tot.html")
# fig_tot

You can also visualize some concrete topics:

In [ ]:
fig_tot_9_10_72 = topic_model.visualize_topics_over_time(topics_over_time, topics=[9, 10, 72])
fig_tot_9_10_72.write_html("figs/fig_tot_9_10_72.html")
# fig_tot_9_10_72